<a href="https://colab.research.google.com/github/chaiyawat19/DataScinceLab/blob/main/Lab9_Gold_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ดึงข้อมูลหุ้น realtime จาก yfinance

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime


start_date = "2024-01-01"
end_date   = end_date = datetime.today().strftime("%Y-%m-%d")

# ตรวจสอบรูปแบบวันที่
try:
    datetime.strptime(start_date, "%Y-%m-%d")
    datetime.strptime(end_date, "%Y-%m-%d")
except ValueError:
    raise ValueError("Date format must be YYYY-MM-DD")

# =========================
# ดึงข้อมูลทองคำ
# =========================
df = yf.download(
    "GC=F",          # Gold Futures
    start=start_date,
    end=end_date,
    interval="1d"
)

if df.empty:
    print("No data found in this date range")
else:
    df = df[["Open","High","Low","Close","Volume"]]
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    df.columns = df.columns.get_level_values(0)
    df.reset_index(inplace=True)

df

แบ่งข้อมูล Train - Test ตามช่วงปี

In [ ]:
df_train = df[
    (df["Date"] >= "2024-01-01") &
    (df["Date"] <= "2024-12-31")
]

# Test = ปี 2025 ถึงวันนี้
df_test = df[
    (df["Date"] >= "2025-01-01")
]

# =========================
# 5) ตรวจสอบผลลัพธ์
# =========================

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
df_train.drop(['Date'],axis=1, inplace=True)

In [ ]:
df_test.drop(['Date'],axis=1, inplace=True)

พยากรณ์ด้วย MLPRegressor

In [ ]:
from sklearn.neural_network import MLPRegressor #output = real value


train_data = df_train.drop(['Close'],axis=1)
train_y = df_train.Close

test_data = df_test.drop(['Close'],axis=1)
test_y = df_test.Close

# Create MLP -> ANN
ann_model = MLPRegressor()

# Train MLP -> ANN Classifer
ann_model.fit(train_data,train_y)

# Test MLP
y_predict = ann_model.predict(test_data)
y_predict

In [ ]:
df_test['Close_predict'] = y_predict
df_test

In [ ]:
from sklearn.metrics import mean_absolute_error
import numpy as np
print(ann_model.score(test_data, test_y))
print(mean_absolute_error(test_y, y_predict))

In [ ]:
from sklearn.metrics import mean_squared_error
from math import sqrt
error = sqrt(mean_squared_error(test_y,y_predict))
error

In [ ]:
d = abs(test_y - y_predict) ## Mean Absolute Error
mse_f = np.mean(d**2)
mae_f = np.mean(abs(d))
rmse_f = np.sqrt(mse_f)
r2_f = 1-(sum(d**2)/sum((test_y-np.mean(test_y))**2))
acc = 100-((100*d)/test_y)

print("Results by manual calculation:")
print("MAE:",mae_f)
print("MSE:", mse_f)
print("RMSE:", rmse_f)
print("R-Squared:", r2_f)
print("%acc:", np.mean(acc))

Save Model

In [ ]:
import pickle

filename = r'/content/gold_ann_model.pkl'
pickle.dump(ann_model, open(filename, 'wb'))